Para facilitar la interacción con Hadoop y la realización de tareas, se ha dispuesto un servidor de cuadernos Jupyter. En él puedes ejecutar, no sólo código Python, sino interaccionar directamente con el sistema operativo a través de comandos.

Por defecto, Jupyter interpreta cada bloque de código como sentencias Python. Sin embargo, si la línea comienza con un símbolo de cierre de exclamación (!), lo enviará al intérprete de órdenes del sistema operativo. En los bloques en que haya varias líneas, basta con poner %%bash al principio del bloque. Mientras esté ejecutándose un bloque, el icono (favicon) de la web pasará a ser un reloj de arena.

# Variables de entorno y directorio de instalación

Como sabes, en los sistemas operativos, existen las denominadas variables de entorno. Una de ellas es la variable PATH, que guarda una serie de rutas a carpetas que contienen ejecutables, y que son examinadas si el ejecutable que trata de invocarse no se encuentra en la carpeta actual. Tener bien configurada esta variable es una buena práctica.

Ejecuta el siguiente comando para observar que, además de la ruta a los binarios de Java, encuentras la de los de Hadoop y otros proyectos Apache relacionados.

In [1]:
! echo $PATH | tr ':' '\n' | grep -E 'java|app'


/usr/lib/jvm/java-8-openjdk-amd64/bin
/app/hadoop-3.3.1/bin
/app/hadoop-3.3.1/sbin
/app/apache-hive-3.1.2-bin/bin
/app/pig-0.17.0/bin
/app/apache-flume-1.9.0-bin/bin
/app/sqoop-1.4.7.bin__hadoop-2.6.0/bin


Otras variables de entorno son usadas por determinadas aplicaciones. Por ejemplo la variable JAVA_HOME es utilizada por aquellos programas que necesitan la JVM, para hacer referencia a esta. En el siguiente ejemplo se muestra dicha ruta.

In [2]:
! echo $JAVA_HOME

/usr/lib/jvm/java-8-openjdk-amd64


**Ejercicio 1:**
- Muestra el valor de la variable HADOOP_HOME
- Muestra el contenido (comando ls, que no requiere exclamación), de dicho directorio
- Muestra el contenido de la subcarpeta bin dentro del directorio de Hadoop, y observa los ejecutables de los que consta.
- Muesra la lista de aquellos ficheros de configuración de la subcarpeta etc/hadoop, que tengan formato xml (usa el comodín *).
- Muestra el conrtenido (comando cat) de los ficheros core-site.xml (configuración general del clúster) y hdfs-site.xml (parámetros generales del sistema de ficheros distribuido).
- Muestra la lista de secciones del manual de hadoop, situado en el subdirectorio share/doc/hadoop (puedes encontrarlo también en la sección Documentation de https://hadoop.apache.org/).

In [22]:
%%bash
#Realiza el ejercicio en este bloque:
echo "-----variable HADOOP_HOME-----"
echo $HADOOP_HOME
echo "----contenido HADOOP_HOME------"
ls $HADOOP_HOME
echo "----contenido HADOOP_HOME/bin------"
ls $HADOOP_HOME/bin
echo "-----contenido ect/hadoop .xml-----"
ls $HADOOP_HOME/etc/hadoop/*.xml
echo "-----contenido archivos-------"
cat $HADOOP_HOME/etc/hadoop/core-site.xml
echo "-------siguiente--------"
cat $HADOOP_HOME/etc/hadoop/hdfs-site.xml
echo "-----secciones del manual------"
ls -R $HADOOP_HOME/share/doc/hadoop

-----variable HADOOP_HOME-----
/app/hadoop-3.3.1
----contenido HADOOP_HOME------
LICENSE-binary
LICENSE.txt
NOTICE-binary
NOTICE.txt
README.txt
bin
etc
include
lib
libexec
licenses-binary
logs
sbin
share
----contenido HADOOP_HOME/bin------
container-executor
hadoop
hadoop.cmd
hdfs
hdfs.cmd
mapred
mapred.cmd
oom-listener
test-container-executor
yarn
yarn.cmd
-----contenido ect/hadoop .xml-----
/app/hadoop-3.3.1/etc/hadoop/capacity-scheduler.xml
/app/hadoop-3.3.1/etc/hadoop/core-site.xml
/app/hadoop-3.3.1/etc/hadoop/hadoop-policy.xml
/app/hadoop-3.3.1/etc/hadoop/hdfs-rbf-site.xml
/app/hadoop-3.3.1/etc/hadoop/hdfs-site.xml
/app/hadoop-3.3.1/etc/hadoop/httpfs-site.xml
/app/hadoop-3.3.1/etc/hadoop/kms-acls.xml
/app/hadoop-3.3.1/etc/hadoop/kms-site.xml
/app/hadoop-3.3.1/etc/hadoop/mapred-site.xml
/app/hadoop-3.3.1/etc/hadoop/yarn-site.xml
-----contenido archivos-------
<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<!--
  Licensed under the

# Ejemplos de MapReduce y comandos básicos de manejo del sistema de ficheros distribuido

Como recordarás, el patrón MapReduce es la manera estándar de realizar consultas en Hadoop. El Map es la consulta que ejecutará cada datanode sobre su parte de los datos. Reduce es la agregación del resultado de dichas consultas, por parte del namenode, para obtener la salida final.

Hadoop proporciona una biblioteca que permite implementar tus propios MapReduce en Java. Además, también incluye una serie de ejemplos en su fichero Jar hadoop-mapreduce-examples.

In [5]:
! ls $HADOOP_HOME/share/hadoop/mapreduce

hadoop-mapreduce-client-app-3.3.1.jar
hadoop-mapreduce-client-common-3.3.1.jar
hadoop-mapreduce-client-core-3.3.1.jar
hadoop-mapreduce-client-hs-3.3.1.jar
hadoop-mapreduce-client-hs-plugins-3.3.1.jar
hadoop-mapreduce-client-jobclient-3.3.1-tests.jar
hadoop-mapreduce-client-jobclient-3.3.1.jar
hadoop-mapreduce-client-nativetask-3.3.1.jar
hadoop-mapreduce-client-shuffle-3.3.1.jar
hadoop-mapreduce-client-uploader-3.3.1.jar
hadoop-mapreduce-examples-3.3.1.jar
jdiff
lib-examples
sources


Para poder probar alguno de los ejemplos, necesitas unos datos de entrada. Puede usarse una copia de los ficheros de configuración del propio Hadoop, o ficheros descargados de algún repositorio de conjuntos de datos (datasets), como https://www.kaggle.com/datasets

Para este ejemplo vamos a descargar simples ficheros de texto. Para ello, puedes usar el comando wget, utilizar curl; o bien instalar alguna otra herramienta alternativa como axel.

El siguiente ejemplo incluye no solo la descarga de un fichero, sino también su renombrado.

In [23]:
%%bash

mkdir Descargas
wget -O Descargas/Don_Quijote.txt https://www.gutenberg.org/ebooks/2000.txt.utf-8 --no-check-certificate

--2026-10-22 11:06:06--  https://www.gutenberg.org/ebooks/2000.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
  Self-signed certificate encountered.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/2000/pg2000.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-10-22 11:06:07--  https://www.gutenberg.org/cache/epub/2000/pg2000.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 2225803 (2.1M) [text/plain]
Saving to: ‘Descargas/Don_Quijote.txt’

     0K .......... .......... .......... .......... ..........  2%  180K 12s
    50K .......... .......... .......... .......... ..........  4%  358K 9s
   100K .......... .......... .......... .......... ..........  6% 18.7M 6s
   150K .......... .......... .....

El ejemplo anterior ha creado una carpeta de descargas, y en ella ha descargado el fichero de texto. Sin embargo, esta descarga se ha realizado en el sistema de ficheros del almacenamiento propio del nodo; no en el sistema de ficheros distribuido (entre los datanodes).

Para manejar dicho sistema de ficheros hacemos uso del intérprete hdfs, y concretamente, de su comando dfs. Recuerda que puedes buscar la ayuda de cualquier comando con -help. Observa que en el siguiente bloque se muestra la ayuda de dfs, que incluye parámetros que seguro que te suenan: cat, chmod, cp, df, ls, rm, etc.

Efectivamente, las opciones indicadas anteriormente comparten nombre con los comandos POSIX con los que estás acostumbrado a manejar los sistemas de ficheros. Estas opciones son sus equivalentes, y cumplen el mismo propósito.


In [25]:
%%bash

#hdfs -help
hdfs dfs -help
# hdfs dfs -help ls

Usage: hadoop fs [generic options]
	[-appendToFile <localsrc> ... <dst>]
	[-cat [-ignoreCrc] <src> ...]
	[-checksum [-v] <src> ...]
	[-chgrp [-R] GROUP PATH...]
	[-chmod [-R] <MODE[,MODE]... | OCTALMODE> PATH...]
	[-chown [-R] [OWNER][:[GROUP]] PATH...]
	[-concat <target path> <src path> <src path> ...]
	[-copyFromLocal [-f] [-p] [-l] [-d] [-t <thread count>] <localsrc> ... <dst>]
	[-copyToLocal [-f] [-p] [-ignoreCrc] [-crc] <src> ... <localdst>]
	[-count [-q] [-h] [-v] [-t [<storage type>]] [-u] [-x] [-e] [-s] <path> ...]
	[-cp [-f] [-p | -p[topax]] [-d] <src> ... <dst>]
	[-createSnapshot <snapshotDir> [<snapshotName>]]
	[-deleteSnapshot <snapshotDir> <snapshotName>]
	[-df [-h] [<path> ...]]
	[-du [-s] [-h] [-v] [-x] <path> ...]
	[-expunge [-immediate] [-fs <path>]]
	[-find <path> ... <expression> ...]
	[-get [-f] [-p] [-ignoreCrc] [-crc] <src> ... <localdst>]
	[-getfacl [-R] <path>]
	[-getfattr [-R] {-n name | -d} [-e en] <path>]
	[-getmerge [-nl] [-skip-empty-file] <src> <localdst>]

La siguiente línea usa precisamente la opción -ls para mostrar el contenido del sistema de ficheros distribuido (desde la raíz), de manera recursiva (-R).

In [26]:
%%bash

hdfs dfs -ls -R /

drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp
drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp/hadoop-yarn
drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp/hadoop-yarn/staging
drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp/hadoop-yarn/staging/history
drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp/hadoop-yarn/staging/history/done
drwxrwxrwx   - root supergroup          0 2026-10-22 10:40 /tmp/hadoop-yarn/staging/history/done_intermediate
drwx-wx-wx   - root supergroup          0 2026-10-22 10:40 /tmp/hive
drwx-wx-wx   - root supergroup          0 2026-10-22 10:40 /tmp/hive/_resultscache_
drwx------   - root supergroup          0 2026-10-22 10:40 /tmp/hive/_resultscache_/results-b2ecbefb-e524-46b0-9d45-2e5fe0353b67
drwx------   - root supergroup          0 2026-10-22 10:41 /tmp/hive/root
drwx------   - root supergroup          0 2026-10-22 10:41 /tmp/hive/root/19fa4248-8d9f-4b3b-ae8b-f3a6cafc772b
drwx------

La siguiente línea crea una carpeta (mkdir). Al no indicarse una ruta absoluta, la creación se realiza en el propio directorio del usuario. 

In [27]:
%%bash

hdfs dfs -mkdir Biblioteca
hdfs dfs -ls

Found 1 items
drwxr-xr-x   - root supergroup          0 2026-10-22 11:16 Biblioteca


A continuación, se sube el fichero de texto del sistema de ficheros local al sistema distribuido, mediante la opción -put. Para bajar, se usa otro término propio del protocolo HTTP, como es -get.

Ten en cuenta que -put no sobreescribe si un fichero homónimo ya existe en el destino, a no ser que se acompañe de la opción -f.

Otra opción parecida es -moveFromLocal, que elimina el archivo original tras subirlo al sistema distribuido.

In [30]:
%%bash

hdfs dfs -put -f Descargas/Don_Quijote.txt Biblioteca
hdfs dfs -ls Biblioteca

Time Stamp               Iteration#  Bytes Already Moved  Bytes Left To Move  Bytes Being Moved  NameNode
Oct 22, 2026 11:24:37 AM          0                  0 B                 0 B                0 B                  0  hdfs://namenode:8020
The cluster is balanced. Exiting...
Oct 22, 2026 11:24:37 AM Balancing took 1.054 seconds


2026-10-22 11:24:36,808 INFO balancer.Balancer: namenodes  = [hdfs://namenode:8020]
2026-10-22 11:24:36,811 INFO balancer.Balancer: parameters = Balancer.BalancerParameters [BalancingPolicy.Node, threshold = 10.0, max idle iteration = 5, #excluded nodes = 0, #included nodes = 0, #source nodes = 0, #blockpools = 0, run during upgrade = false]
2026-10-22 11:24:36,811 INFO balancer.Balancer: included nodes = []
2026-10-22 11:24:36,811 INFO balancer.Balancer: excluded nodes = []
2026-10-22 11:24:36,811 INFO balancer.Balancer: source nodes = []
2026-10-22 11:24:36,814 INFO balancer.NameNodeConnector: getBlocks calls for hdfs://namenode:8020 will be rate-limited to 20 per second
2026-10-22 11:24:37,642 INFO balancer.Balancer: dfs.namenode.get-blocks.max-qps = 20 (default=20)
2026-10-22 11:24:37,642 INFO balancer.Balancer: dfs.balancer.movedWinWidth = 5400000 (default=5400000)
2026-10-22 11:24:37,642 INFO balancer.Balancer: dfs.balancer.moverThreads = 1000 (default=1000)
2026-10-22 11:24:37,6

Accede ahora a la dirección web del namenode (nodo maestro): http://localhost:9870 (aunque en core-site.xml figura el puerto 8020 para el namenode, ese es el que utiliza para la conexión con el servicio HDFS de los datanodes, no para mostrar la interfaz al usuario).

Una vez ahí, pulsa "Datanodes" en el menú; y fíjate en los 3 nodos que tienen el mismo espacio ocupado.

¿A qué se corresponde ese espacio?

In [31]:
! ls -lh Descargas

total 2.2M
-rw-r--r-- 1 root root 2.2M Oct  1 17:31 Don_Quijote.txt


Efectivamente, el tamaño coincide con el fichero que acabas de subir. Pero, ¿recuerdas por qué está en los tres nodos?

**Escribe tu respuesta aquí:** 
porque la información se guarda por triplicado en nodos diferentes

Por defecto, son 3 nodos; aunque este valor puede cambiarse en el hdfs-site.xml, mediante la propiedad dfs.replication.

Pero no busques esa propiedad en ese fichero. No ha sido establecida, y por eso Hadoop usa el valor por defecto.

**Ejercicio 2:** vas a replicar la subida de ficheros al sistema distribuido. Para ello, descarga algunos ficheros más, usando el comando que prefieras. Se te proporcionan los enlaces:

- Novelas ejemplares: https://www.gutenberg.org/files/61202/61202-0.txt
- Los entremeses: https://www.gutenberg.org/files/57955/57955-8.txt
- Viaje al Parnaso, La Numancia, y el Trato de Argel: https://www.gutenberg.org/files/16110/16110-8.txt

Después, súbelos a la misma carpeta a la que se subió el anterior fichero.

In [50]:
%%bash
#Realiza el ejercicio en este bloque:
hdfs balancer

Time Stamp               Iteration#  Bytes Already Moved  Bytes Left To Move  Bytes Being Moved  NameNode
Oct 22, 2026 11:40:11 AM          0                  0 B                 0 B                0 B                  0  hdfs://namenode:8020
The cluster is balanced. Exiting...
Oct 22, 2026 11:40:11 AM Balancing took 1.065 seconds


2026-10-22 11:40:10,876 INFO balancer.Balancer: namenodes  = [hdfs://namenode:8020]
2026-10-22 11:40:10,880 INFO balancer.Balancer: parameters = Balancer.BalancerParameters [BalancingPolicy.Node, threshold = 10.0, max idle iteration = 5, #excluded nodes = 0, #included nodes = 0, #source nodes = 0, #blockpools = 0, run during upgrade = false]
2026-10-22 11:40:10,880 INFO balancer.Balancer: included nodes = []
2026-10-22 11:40:10,880 INFO balancer.Balancer: excluded nodes = []
2026-10-22 11:40:10,880 INFO balancer.Balancer: source nodes = []
2026-10-22 11:40:10,884 INFO balancer.NameNodeConnector: getBlocks calls for hdfs://namenode:8020 will be rate-limited to 20 per second
2026-10-22 11:40:11,723 INFO balancer.Balancer: dfs.namenode.get-blocks.max-qps = 20 (default=20)
2026-10-22 11:40:11,723 INFO balancer.Balancer: dfs.balancer.movedWinWidth = 5400000 (default=5400000)
2026-10-22 11:40:11,724 INFO balancer.Balancer: dfs.balancer.moverThreads = 1000 (default=1000)
2026-10-22 11:40:11,7

**Ejercicio 3:** familiarizaté con hdfs dfs. Sabiendo lo que sabes de comandos de Linux. Crea una carpeta Cervantes dentro de Biblioteca, y mueve ahí los ficheros de texto que hay dentro de la propia Biblioteca.

In [51]:
%%bash
#Realiza el ejercicio en este bloque:
hdfs dfs -mkdir Biblioteca/Cervantes
hdfs dfs -mv Biblioteca/*.txt Biblioteca/Cervantes

También a través de la UI del namenode (http://localhost:9870), puedes acceder al árbol de directorios del sistema de ficheros distribuido. En el menú pulsa sobre "Utilities", y elige la opción "Browse the file system". A partir de ahí, pulsa en el nombre de cada carpeta para llegar hasta la denominada Cervantes.

Ahora, a partir de estos archivos, vamos a realizar una búsqueda usando MapReduce. Específicamente, ya que nos gusta nuestra ciudad, vamos a comprobar si en alguna de esas obras se hace referencia a Salamanca, y en cuantas ocasiones se menciona.

La ejecución de MapReduce no se hace utilizando hdfs, pues este programa está dedicado a la gestión del sistema de ficheros distribuido. Haremos uso del CLI hadoop. Y más concretamente de su comando jar.

Este comando recibe como argumentos:
- La ruta al fichero jar (Java ARchive) que contiene el bytecode de la implementación del programa MapReduce.
- La clase del jar que se desea ejecutar (grep).
- Los argumentos que el programa en cuestión necesite. En este caso:
    - Ruta al fichero o ficheros que contengan la información de entrada.
    - Ruta de la carpeta donde se situará el resultado final que se obtiene tras ejecutarse el Recuce en el namenode.
    
Durante la ejecución del MapReduce, se muestran una serie de estadísticas: el progreso, la cantidad de salidas intermedias (map), el tiempo de procesamiento, etc.
    
Una vez finalice la ejecución del MapReduce, puede consultarse el resultado con un cat al contenido de la carpeta del resultado. Las veces que se ha encontrado el término buscado figurarán delante del propio término. Este es el clásico ejemplo de contador de ocurrencias de palabras con Hadoop.

In [3]:
%env HADOOP_VERSION=3.3.1

env: HADOOP_VERSION=3.3.1


In [56]:
%%bash


mkdir Resultados
hadoop jar /app/hadoop-3.3.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.1.jar grep Biblioteca/Cervantes Resultados/Salamanca_Cervantes "Salamanca"
hdfs dfs -cat Resultados/Salamanca_Cervantes/*

63	Salamanca


mkdir: cannot create directory ‘Resultados’: File exists
2026-10-22 12:03:55,110 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-22 12:03:55,470 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1792658436998_0001
2026-10-22 12:03:55,689 INFO input.FileInputFormat: Total input files to process : 4
2026-10-22 12:03:55,766 INFO mapreduce.JobSubmitter: number of splits:4
2026-10-22 12:03:55,895 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1792658436998_0001
2026-10-22 12:03:55,895 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-22 12:03:56,055 INFO conf.Configuration: resource-types.xml not found
2026-10-22 12:03:56,055 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-22 12:03:56,577 INFO impl.YarnClientImpl: Submitted application application_1792658436998_0001
2026-10-22 12:03:56,602 INFO mapreduce.Job: 

Accede a la interfaz del gestor de recursos (YARN) de Hadoop: http://localhost:8088

Ahí aparecerá registrado el trabajo que acabas de ejecutar, y si finalizó de manera exitosa.

**Ejercicio 4:** lleva a cabo la búsqueda de otra palabra (a tu elección), siguiendo los pasos anteriores.

In [3]:
%%bash
#Realiza el ejercicio en este bloque:
hadoop jar /app/hadoop-3.3.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.1.jar grep Biblioteca/Cervantes Resultados/Salamanca_Cervantes "Salamanca"
hdfs dfs -cat Resultados_Ej4/Salamanca_Cervantes/*

2026-10-26 13:38:39,063 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-26 13:38:39,320 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793018179821_0003
2026-10-26 13:38:39,497 INFO input.FileInputFormat: Total input files to process : 4
2026-10-26 13:38:39,559 INFO mapreduce.JobSubmitter: number of splits:4
2026-10-26 13:38:39,643 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1793018179821_0003
2026-10-26 13:38:39,643 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-26 13:38:39,748 INFO conf.Configuration: resource-types.xml not found
2026-10-26 13:38:39,748 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-26 13:38:39,802 INFO impl.YarnClientImpl: Submitted application application_1793018179821_0003
2026-10-26 13:38:39,834 INFO mapreduce.Job: The url to track the job: http://yarnmaster:8088/proxy/ap

CalledProcessError: Command 'b'#Realiza el ejercicio en este bloque:\nhadoop jar /app/hadoop-3.3.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.1.jar grep Biblioteca/Cervantes Resultados/Salamanca_Cervantes "Salamanca"\nhdfs dfs -cat Resultados_Ej4/Salamanca_Cervantes/*\n'' returned non-zero exit status 1.

Es posible buscar varias palabras diferentes. Para ello se separan con la barra vertical (|):

In [11]:
%%bash
mkdir Resultados/CiudadesCyl_Cervantes
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar grep Biblioteca/Cervantes/*.txt Resultados/CiudadesCyL_Cervantes "Quijote|Alonso Quijano|Sancho|Dulcinea|Rocinante"
hdfs dfs -cat Resultados/CiudadesCyL_Cervantes/*

2241	Quijote
2185	Sancho
287	Dulcinea
206	Rocinante
5	Alonso Quijano


mkdir: cannot create directory ‘Resultados/CiudadesCyl_Cervantes’: File exists
2026-10-26 14:04:07,834 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-26 14:04:08,176 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793018179821_0004
2026-10-26 14:04:08,366 INFO input.FileInputFormat: Total input files to process : 4
2026-10-26 14:04:08,446 INFO mapreduce.JobSubmitter: number of splits:4
2026-10-26 14:04:08,565 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1793018179821_0004
2026-10-26 14:04:08,565 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-26 14:04:08,694 INFO conf.Configuration: resource-types.xml not found
2026-10-26 14:04:08,695 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-26 14:04:08,755 INFO impl.YarnClientImpl: Submitted application application_1793018179821_0004
2026-10-26 14:04:08,78

**Ejercicio 5:** busca, únicamente en el Quijote, cuantas veces se menciona a cada uno de los personajes principales:
- Quijote
- Alonso Quijano (nombre real del protagonista)
- Sancho
- Dulcinea
- Rocinante

In [14]:
%%bash
# 1. Usar -r para borrar el directorio completo de forma recursiva
hdfs dfs -rm -r Resultados/ejercicio_5

# 2. Ejecutar el ejemplo de grep de Hadoop
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar grep Biblioteca/Cervantes/*.txt Resultados/ejercicio_5 "Quijote|Alonso Quijano|Sancho|Dulcinea|Rocinante"

# 3. Mostrar los resultados
hdfs dfs -cat Resultados/ejercicio_5/*

Deleted Resultados/ejercicio_5
2241	Quijote
2185	Sancho
287	Dulcinea
206	Rocinante
5	Alonso Quijano


2026-10-26 14:07:10,872 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-26 14:07:11,087 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793018179821_0008
2026-10-26 14:07:11,260 INFO input.FileInputFormat: Total input files to process : 4
2026-10-26 14:07:11,324 INFO mapreduce.JobSubmitter: number of splits:4
2026-10-26 14:07:11,409 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1793018179821_0008
2026-10-26 14:07:11,409 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-26 14:07:11,538 INFO conf.Configuration: resource-types.xml not found
2026-10-26 14:07:11,539 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-26 14:07:11,579 INFO impl.YarnClientImpl: Submitted application application_1793018179821_0008
2026-10-26 14:07:11,607 INFO mapreduce.Job: The url to track the job: http://yarnmaster:8088/proxy/ap

**Ejercicio 6:** Está consulta que has realizado, buscando no una única palabra, sino varias, se basa en expresiones regulares (regexp).
    
Es muy posible que ya las hayas usado previamente, en anteriores estudios, o en el trabajo. De no ser así, puedes aprender los elementos más utilizados, a través de este educativo juego: https://manzdev.github.io/regex-people/

Después, realiza las siguientes búsquedas:
- Cuantas personas reciben el tratamiento de Don o Doña.
- Palabras que acaben en cijo.
- Palabras que empiecen y terminen por la letra "e".
- Palabras de más de 14 letras.

In [17]:
%%bash
#Realiza el ejercicio en este bloque:

hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar grep Biblioteca/Cervantes/*.txt Resultados/E_Quijote "(E| e)[a-z]+e"

hdfs dfs -cat Resultados/E_Quijote/*

218	 ene
204	 eje
194	 ese
156	 efe
126	 ere
119	 ele
46	 ete
35	 exe
22	Ese
15	Ene
6	Ere
6	Ele
4	 ece
4	Eje
3	Exe
2	Eme
2	Ete
1	 eme
1	Efe


2026-10-29 11:04:43,796 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-29 11:04:44,001 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793266726428_0009
2026-10-29 11:04:44,150 INFO input.FileInputFormat: Total input files to process : 4
2026-10-29 11:04:44,216 INFO mapreduce.JobSubmitter: number of splits:4
2026-10-29 11:04:44,300 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1793266726428_0009
2026-10-29 11:04:44,300 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-29 11:04:44,435 INFO conf.Configuration: resource-types.xml not found
2026-10-29 11:04:44,435 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-29 11:04:44,473 INFO impl.YarnClientImpl: Submitted application application_1793266726428_0009
2026-10-29 11:04:44,494 INFO mapreduce.Job: The url to track the job: http://yarnmaster:8088/proxy/ap

Como hemos mencionado, Grep es una clase que se encuentra dentro del JAR de ejemplo. Pero si miramos el interior de ese JAR, vamos a ver muchas más clases. Una gran parte de ellas están relacionadas con el análisis de ficheros de texto, pero no todas lo están.

In [18]:
%%bash

jar tf $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar

META-INF/
META-INF/MANIFEST.MF
org/
org/apache/
org/apache/hadoop/
org/apache/hadoop/examples/
org/apache/hadoop/examples/pi/
org/apache/hadoop/examples/pi/math/
org/apache/hadoop/examples/dancing/
org/apache/hadoop/examples/terasort/
META-INF/LICENSE.txt
META-INF/NOTICE.txt
org/apache/hadoop/examples/ExampleDriver.class
org/apache/hadoop/examples/DBCountPageView$AccessRecord.class
org/apache/hadoop/examples/BaileyBorweinPlouffe$Fraction.class
org/apache/hadoop/examples/WordMedian$WordMedianReducer.class
org/apache/hadoop/examples/WordStandardDeviation.class
org/apache/hadoop/examples/WordStandardDeviation$WordStandardDeviationReducer.class
org/apache/hadoop/examples/Join.class
org/apache/hadoop/examples/pi/Util.class
org/apache/hadoop/examples/pi/SummationWritable$ArithmeticProgressionWritable.class
org/apache/hadoop/examples/pi/Parser.class
org/apache/hadoop/examples/pi/DistSum$ReduceSide$PartitionMapper.class
org/apache/hadoop/examples/pi/math/Montgomery.class
org/apache/hadoop/exam

In [19]:
%%bash

# Obtener estimación del valor del número pi.
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar pi 6 5000

Number of Maps  = 6
Samples per Map = 5000
Wrote input for Map #0
Wrote input for Map #1
Wrote input for Map #2
Wrote input for Map #3
Wrote input for Map #4
Wrote input for Map #5
Starting Job
Job Finished in 14.008 seconds
Estimated value of Pi is 3.14146666666666666667


2026-10-29 11:28:14,438 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-29 11:28:14,694 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793266726428_0010
2026-10-29 11:28:14,781 INFO input.FileInputFormat: Total input files to process : 6
2026-10-29 11:28:14,827 INFO mapreduce.JobSubmitter: number of splits:6
2026-10-29 11:28:14,942 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1793266726428_0010
2026-10-29 11:28:14,942 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-29 11:28:15,059 INFO conf.Configuration: resource-types.xml not found
2026-10-29 11:28:15,060 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-29 11:28:15,105 INFO impl.YarnClientImpl: Submitted application application_1793266726428_0010
2026-10-29 11:28:15,134 INFO mapreduce.Job: The url to track the job: http://yarnmaster:8088/proxy/ap

In [20]:
%%bash

# Saber qué palabras aparecen con más frecuencia en un texto.
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar wordcount Biblioteca/Cervantes/Novelas_ejemplares.txt Resultados/Contador_Novelas_Ejemplares
hdfs dfs -cat Resultados/Contador_Novelas_Ejemplares/* | sort -n -r -k2 | head -n 200 | nl

2026-10-29 11:28:29,684 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-29 11:28:29,900 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793266726428_0011
2026-10-29 11:28:30,059 INFO mapreduce.JobSubmitter: Cleaning up the staging area /tmp/hadoop-yarn/staging/root/.staging/job_1793266726428_0011
org.apache.hadoop.mapreduce.lib.input.InvalidInputException: Input path does not exist: hdfs://namenode:8020/user/root/Biblioteca/Cervantes/Novelas_ejemplares.txt
	at org.apache.hadoop.mapreduce.lib.input.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:340)
	at org.apache.hadoop.mapreduce.lib.input.FileInputFormat.listStatus(FileInputFormat.java:279)
	at org.apache.hadoop.mapreduce.lib.input.FileInputFormat.getSplits(FileInputFormat.java:404)
	at org.apache.hadoop.mapreduce.JobSubmitter.writeNewSplits(JobSubmitter.java:310)
	at org.apache.hadoop.m

In [21]:
%%bash

#Longitud media de las palabras de las obras cervantinas descargadas.
hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-$HADOOP_VERSION.jar wordmean Biblioteca/Cervantes/*.txt Resultados/Media_Cervantes
hdfs dfs -cat Resultados/Media_Cervantes/*


The mean is: 4.478010647042014
count	678874
length	3040005


2026-10-29 11:28:32,615 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmaster/172.18.0.3:8032
2026-10-29 11:28:32,833 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1793266726428_0012
2026-10-29 11:28:33,004 INFO input.FileInputFormat: Total input files to process : 4
2026-10-29 11:28:33,075 INFO mapreduce.JobSubmitter: number of splits:4
2026-10-29 11:28:33,165 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1793266726428_0012
2026-10-29 11:28:33,165 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-29 11:28:33,303 INFO conf.Configuration: resource-types.xml not found
2026-10-29 11:28:33,303 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-29 11:28:33,345 INFO impl.YarnClientImpl: Submitted application application_1793266726428_0012
2026-10-29 11:28:33,380 INFO mapreduce.Job: The url to track the job: http://yarnmaster:8088/proxy/ap